In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time
import random

In [2]:
BASE_URL   = 'https://kolesa.kz'
SEARCH_URL = 'https://kolesa.kz/cars/'
HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/120.0.0.0 Safari/537.36'
    ),
    'Accept-Language': 'ru-RU,ru;q=0.9,en;q=0.8',
}
TARGET_PAGES = 3

In [3]:
def get_page(url: str, retries: int = 3):
    for attempt in range(retries):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=15)
            resp.raise_for_status()
            return BeautifulSoup(resp.text, 'html.parser')
        except requests.exceptions.HTTPError as e:
            print(f'  [HTTP error] {e} — attempt {attempt+1}/{retries}')
        except requests.exceptions.ConnectionError:
            print(f'  [Connection error] {url} — attempt {attempt+1}/{retries}')
        except requests.exceptions.Timeout:
            print(f'  [Timeout] {url} — attempt {attempt+1}/{retries}')
        time.sleep(2 ** attempt)
    return None

In [4]:
def parse_listing(card) -> dict | None:
    try:
        title_el = card.select_one('h5.a-card__title a')
        title    = title_el.get_text(strip=True) if title_el else ''

        href = title_el.get('href', '') if title_el else ''
        link = BASE_URL + href if href.startswith('/') else href

        price_el  = card.select_one('[data-test="a-card-price"]')
        raw_price = price_el.get_text(strip=True) if price_el else '0'

        city_el = card.select_one('[data-test="region"]')
        city    = city_el.get_text(strip=True) if city_el else 'N/A'

        desc_el = card.select_one('p.a-card__description')
        desc    = desc_el.get_text(strip=True) if desc_el else ''

        year_match = re.search(r'\b(19|20)\d{2}\b', desc)
        year = int(year_match.group()) if year_match else None

        engine_match = re.search(r'[\d.]+\s*л', desc)
        engine = engine_match.group().strip() if engine_match else ''

        mileage_match = re.search(r'пробегом\s*([\d\s]+)\s*км', desc)
        raw_mileage   = mileage_match.group(1).strip() + ' km' if mileage_match else ''

        body_match = re.search(r'\d{4}\s*г\.,\s*(?:Б/у|Новый|б/у)?\s*([^,]+)', desc)
        body = body_match.group(1).strip() if body_match else ''

        return {
            'title':       title,
            'raw_price':   raw_price,
            'year':        year,
            'city':        city,
            'engine':      engine,
            'body_type':   body,
            'raw_mileage': raw_mileage,
            'link':        link,
        }
    except Exception as e:
        print(f'  [parse error] {e}')
        return None

In [5]:
def scrape(pages: int = TARGET_PAGES) -> list[dict]:
    all_listings = []
    for page_num in range(1, pages + 1):
        url = SEARCH_URL if page_num == 1 else f'{SEARCH_URL}?page={page_num}'
        print(f'Page {page_num}: {url}')

        soup = get_page(url)
        if soup is None:
            print(f'  Skipping page {page_num}.')
            continue

        cards = soup.select('.a-card') or soup.select('.list-card')
        print(f'  Cards found: {len(cards)}')

        for card in cards:
            listing = parse_listing(card)
            if listing:
                all_listings.append(listing)

        time.sleep(1.5 + random.uniform(0, 0.5))

    return all_listings


raw_data = scrape()
print(f'\nTotal listings collected: {len(raw_data)}')

Page 1: https://kolesa.kz/cars/
  Cards found: 20
Page 2: https://kolesa.kz/cars/?page=2
  Cards found: 20
Page 3: https://kolesa.kz/cars/?page=3
  Cards found: 20

Total listings collected: 60


In [6]:
def clean_price(raw: str) -> int | None:
    digits = re.sub(r'[^\d]', '', raw)
    return int(digits) if digits else None


def clean_mileage(raw: str) -> int | None:
    if not raw:
        return None
    digits = re.sub(r'[^\d]', '', raw)
    if not digits:
        return None
    value = int(digits)
    if 'тыс' in raw.lower() and value < 10_000:
        value *= 1000
    return value

In [7]:
def clean_data(records: list[dict]) -> pd.DataFrame:
    df = pd.DataFrame(records)
    if df.empty:
        return df

    df['price']      = df['raw_price'].apply(clean_price)
    df['mileage_km'] = df['raw_mileage'].apply(clean_mileage)

    df = df.dropna(subset=['price'])
    df['price'] = df['price'].astype(int)

    before = len(df)
    df = df.drop_duplicates(subset=['title', 'price', 'city'])
    print(f'Duplicates removed: {before - len(df)}')

    df['brand'] = df['title'].str.split().str[0]
    df['model'] = df['title'].str.split().str[1:3].str.join(' ')

    cols = ['title','brand','model','price','year','city',
            'engine','body_type','mileage_km','link']
    return df[[c for c in cols if c in df.columns]].reset_index(drop=True)


df = clean_data(raw_data)
print(f'Clean rows: {len(df)}')
df.head()

Duplicates removed: 1
Clean rows: 59


,title,brand,model,price,year,city,engine,body_type,mileage_km,link
0,BYD Tang L,BYD,Tang L,23400000,2025,Шымкент,1.5 л,новый внедорожник,NaN,https://kolesa.kz/a/show/199360179?search_id=5...
1,Toyota Corolla Comfort CVT,Toyota,Corolla Comfort,14690000,2025,Астана,1.6 л,новый седан,NaN,https://kolesa.kz/a/show/211890959?search_id=5...
2,BYD Destroyer 05 Luxury,BYD,Destroyer 05,8200000,2025,Алматы,1.5 л,новый седан,NaN,https://kolesa.kz/a/show/209401486?search_id=5...
3,BYD Song Pro,BYD,Song Pro,13599000,2025,Алматы,1.5 л,новый кроссовер,NaN,https://kolesa.kz/a/show/209591010?search_id=5...
4,BYD Yuan Up Flagship,BYD,Yuan Up,10990000,2026,Шымкент,,новый кроссовер,NaN,https://kolesa.kz/a/show/204542397?search_id=5...


In [8]:
print('Data types:')
print(df.dtypes)
print('\nBasic statistics:')
df[['price','year','mileage_km']].describe().map(lambda x: f'{x:,.0f}')

Data types:
title             str
brand          object
model          object
price           int64
year            int64
city              str
engine            str
body_type         str
mileage_km    float64
link              str
dtype: object

Basic statistics:


,price,year,mileage_km
count,59,59,17
mean,"17,790,937","2,020","128,035"
std,"23,732,957",9,"129,797"
min,"2,000,000","1,993",15
25%,"7,745,000","2,019","49,000"
50%,"11,890,000","2,025","77,000"
75%,"17,445,000","2,025","190,000"
max,"155,323,000","2,026","440,000"


In [9]:
avg_by_year = (
    df.groupby('year')['price']
    .mean()
    .round(0)
    .astype(int)
    .sort_index()
    .reset_index()
    .rename(columns={'price': 'avg_price'})
)

avg_by_year.style \
    .format({'avg_price': '{:,}'}) \
    .background_gradient(subset=['avg_price'], cmap='Blues') \
    .set_caption('Average car price by year')

,year,avg_price
0,1993,"2,475,000"
1,1997,"2,350,000"
2,2001,"2,900,000"
3,2002,"2,325,000"
4,2008,"5,395,000"
5,2010,"4,400,000"
6,2011,"15,350,000"
7,2013,"2,200,000"
8,2015,"4,200,000"
9,2019,"22,985,000"


In [10]:
city_counts = (
    df['city'].value_counts()
    .reset_index()
    .rename(columns={'city': 'city', 'count': 'listings'})
)

print(f'City with most listings: {city_counts.iloc[0]["city"]}')

city_counts.style \
    .bar(subset=['listings'], color='#1F4E79') \
    .set_caption('Listings by city')


City with most listings: Алматы


,city,listings
0,Алматы,18
1,Астана,12
2,Шымкент,4
3,Караганда,4
4,Костанай,4
5,Актобе,3
6,Павлодар,2
7,Атырау,2
8,Актау,1
9,Талгар,1


In [11]:
top3_cheap = df.nsmallest(3, 'price')[['title','price','year','city']]
top3_exp   = df.nlargest(3,  'price')[['title','price','year','city']]

print('Top 3 cheapest:')
display(top3_cheap.style.format({'price': '{:,} ₸'}))

print('Top 3 most expensive:')
display(top3_exp.style.format({'price': '{:,} ₸'}))

Top 3 cheapest:


,title,price,year,city
6,SsangYong Istana,"2,000,000 ₸",2002,Талгар
12,ВАЗ (Lada) Lada 2121,"2,200,000 ₸",2013,Шиели
31,Volkswagen Golf,"2,250,000 ₸",1993,Тараз


Top 3 most expensive:


,title,price,year,city
48,Land Rover Defender OCTA,"155,323,000 ₸",2025,Атырау
9,Infiniti QX80,"88,990,000 ₸",2025,Астана
5,Infiniti QX80,"69,990,000 ₸",2025,Актау


In [12]:
brand_stats = (
    df.groupby('brand').agg(
        listings=('price', 'count'),
        avg_price=('price', 'mean'),
        min_price=('price', 'min'),
        max_price=('price', 'max'),
    )
    .round(0)
    .astype(int)
    .sort_values('listings', ascending=False)
    .reset_index()
)

brand_stats.style \
    .format({'avg_price': '{:,}', 'min_price': '{:,}', 'max_price': '{:,}'}) \
    .background_gradient(subset=['listings'], cmap='Greens') \
    .set_caption('Stats by brand')


,brand,listings,avg_price,min_price,max_price
0,Toyota,8,"18,747,500","5,890,000","32,500,000"
1,Changan,4,"11,845,000","6,990,000","16,890,000"
2,Geely,4,"9,315,000","7,990,000","10,990,000"
3,Haval,4,"12,159,200","7,500,000","15,825,300"
4,BYD,4,"14,047,250","8,200,000","23,400,000"
5,Hyundai,3,"10,233,333","3,200,000","14,000,000"
6,Chevrolet,3,"9,723,333","7,290,000","11,690,000"
7,ГАЗ,3,"7,710,667","2,650,000","14,882,000"
8,Infiniti,2,"79,490,000","69,990,000","88,990,000"
9,Li,2,"25,804,500","21,809,000","29,800,000"


In [13]:
df.to_csv('kolesa_cars.csv', index=False, encoding='utf-8-sig')
print('CSV saved in kolesa_cars.csv')

with pd.ExcelWriter('kolesa_cars.xlsx', engine='openpyxl') as writer:
    df.to_excel(writer, sheet_name='All listings', index=False)
    avg_by_year.to_excel(writer, sheet_name='Price by year', index=False)
    city_counts.to_excel(writer, sheet_name='Cities', index=False)
    brand_stats.to_excel(writer, sheet_name='Brand stats', index=False)

    cheap_out = top3_cheap.copy(); cheap_out['type'] = 'Cheapest'
    exp_out   = top3_exp.copy();   exp_out['type']   = 'Most expensive'
    pd.concat([cheap_out, exp_out]).to_excel(
        writer, sheet_name='Top listings', index=False
    )

print('Excel saved in kolesa_cars.xlsx')
print(f'Total rows in dataset: {len(df)}')


CSV saved in kolesa_cars.csv
Excel saved in kolesa_cars.xlsx
Total rows in dataset: 59
